In [1]:
from sync_workbench.storage.sqlite_store import SQLiteCoreStore

sqlite_path = "workbench.sqlite"

store = SQLiteCoreStore(sqlite_path)

In [2]:
df_dr = store.read_table("DEVICE_RUN")
df_sm = store.read_table("SAMPLE_MAPPING")
df_ste = store.read_table("SAMPLE_TIME_ESTIMATE")

In [3]:
from sync_workbench.sync.mapping import TimelineSelection
subject = "19_MM" #"07_SW"
source_run = "Session-2024-January-15 09-47-41-274452"
target_run = "Session-2024-January-15 09-43-27-126274" #"Session-2023-November-27 13-59-35-723243"
source_device = "kinect_rgb"
target_device = "radar_pc"
source_timeline = "rgb_wallclock_from_pts"
target_timeline = "radar_pc_linear_from_index"

source_selection = TimelineSelection(subject, source_run, source_device, source_timeline)
target_selection = TimelineSelection(subject, target_run, target_device, target_timeline)

rgb_mask = (
    (df_ste["subject_id"] == source_selection.subject_id)
    & (df_ste["run_id"] == source_selection.run_id)
    & (df_ste["device_type"] == source_selection.device_type)
    & (df_ste["timeline_model_id"] == source_selection.timeline_model_id)
)
pc_mask = (
    (df_ste["subject_id"] == target_selection.subject_id)
    & (df_ste["run_id"] == target_selection.run_id)
    & (df_ste["device_type"] == target_selection.device_type)
    & (df_ste["timeline_model_id"] == target_selection.timeline_model_id)
)

rg_ste = df_ste.loc[rgb_mask]
pc_ste = df_ste.loc[pc_mask]

In [4]:
print(rg_ste[['time_value_sec','time_value_datetime','reference_time_datetime']].iloc[0])
print(pc_ste[['time_value_sec','time_value_datetime','reference_time_datetime']].iloc[0])

time_value_sec                                    NaN
time_value_datetime        2024-01-15T09:48:05.445703
reference_time_datetime    2024-01-15T09:48:05.445703
Name: 263864, dtype: object
time_value_sec                                    NaN
time_value_datetime        2024-01-15T09:44:08.296777
reference_time_datetime    2024-01-15T09:44:08.352672
Name: 641137, dtype: object


In [24]:
pc_ste.columns

Index(['subject_id', 'run_id', 'device_type', 'timeline_model_id',
       'sample_index', 'time_value_datetime', 'time_value_sec', 'time_kind',
       'reference_time_datetime', 'residual_ms', 'notes'],
      dtype='str')

In [ ]:
from sync_workbench.core.time_utils import datetime_to_epoch_seconds, utc_now_str

dt_text = rg_ste["time_value_datetime"].fillna("").astype(str).str.strip()
has_datetime = dt_text.ne("") & ~dt_text.str.lower().isin({"nan", "none", "nat"})
rg_ste["time_value_datetime"].map(datetime_to_epoch_seconds).iloc[0]
#first rgb frame seconds: 1705312085.445703

dt_text = pc_ste["time_value_datetime"].fillna("").astype(str).str.strip()
pc_ste["time_value_datetime"].map(datetime_to_epoch_seconds).iloc[0]
#first pc frame seconds: 1705351448.296777

np.float64(1705351448.296777)

In [14]:
#all is_primary==0
#
min_delta_idx = df_sm['predicted_minus_estimated_ms'].argmin()
df_sm.iloc[min_delta_idx]

subject_id                                                        19_MM
mapping_version_id                               rgb_to_pc_initial_v001
source_run_id                   Session-2024-January-15 09-47-41-274452
source_device_type                                           kinect_rgb
source_sample_index                                               34285
target_run_id                   Session-2024-January-15 09-43-27-126274
target_device_type                                             radar_pc
target_sample_index                                                   0
predicted_minus_estimated_ms                         36490717.751026154
rank                                                                  1
is_primary                                                            0
mapping_region_type                                  left_extrapolation
support_status                                              outside_run
confidence_score                                                